In [132]:
from dendropy.simulate import treesim
import dendropy
import treeswift as ts
import msprime
import numpy as np
import collections
import sys

In [133]:
t = dendropy.Tree.get(
    path="../main-num_generations.tre",
    schema="newick",
    preserve_underscores=True
)

In [134]:
samples = []
height = t.max_distance_from_root()
name_to_pop = {}
for ix, node in enumerate(t.leaf_nodes()):
    # st = height - node.distance_from_root()
    # node.edge_length += st
    st = 0
    samples.append(
        msprime.SampleSet(1, ploidy=1, time=st, population=node.taxon.label)
    )
    name_to_pop[ix] = node.taxon.label

In [135]:
initial_size = {}
with open("../population-sizes.tsv", 'r') as f:
    initial_size = dict(map(lambda x : x.strip().split("\t"), f.readlines()[1:]))
initial_size = {k:float(v) for k, v in initial_size.items()}

In [136]:
tree = t.as_string(schema="newick", suppress_rooting=True, unquoted_underscores=True)
demography = msprime.Demography.from_species_tree(tree, initial_size)

In [137]:
demography.sort_events()

In [138]:
for ix, pop in enumerate(demography.populations):
    name_to_pop[ix] = pop.name

In [139]:
SL = 1000
R = 5e-10 # 5e-8

In [140]:
tseq = msprime.sim_ancestry(
        samples=samples,
        ploidy=2,
        demography=demography,
        recombination_rate=R,
        sequence_length=SL,
        random_seed=(hash(1) % 100000),
)
tables = tseq.dump_tables()

The provenance information for the resulting tree sequence is 2.72MB. This is nothing to worry about as provenance is a good thing to have, but if you want to save this memory/storage space you can disable provenance recording by setting record_provenance=False


In [141]:
pop_to_rmult = {}
pop_to_srate = {}
with open("../substitution-rate.tsv", 'r') as f:
    ix = 0
    mean_srate = 0
    for l in f:
        if ix > 0:
            name, srate = l.strip().split("\t")
            pop_to_srate[name] = float(srate)
            mean_srate += float(srate)
        ix = ix + 1
    mean_srate /= ix
    for name, srate in pop_to_srate.items():
        pop_to_rmult[name] = srate/mean_srate

In [142]:
tmain = ts.read_tree_newick("../main-num_generations.tre")
pop_to_stime = {}
pop_to_time = {}

for nd in tmain.traverse_levelorder(internal=True, leaves=True):
    l = nd.get_edge_length()
    if l is None:
        l = 0
    pop_to_time[nd.get_label()] = tmain.extract_subtree(nd).height() - l
    
for nd in tmain.traverse_postorder(internal=True, leaves=False):
    for nd_child in nd.child_nodes():
        nd_child.set_edge_length(nd_child.get_edge_length() * pop_to_rmult[nd_child.get_label()])
    
h = tmain.height()
for nd, dist in tmain.distances_from_root(leaves=True, internal=True, unlabeled=True, weighted=True):
    hp = h - dist
    pop_to_stime[nd.get_label()] = hp

In [143]:
def scale_time(name, time):
    pop = name_to_pop[name]
    return pop_to_stime[pop] + (time - pop_to_time[pop]) * pop_to_rmult[pop]

In [144]:
stime_l = [scale_time(i.population, i.time) for i in tables.nodes]
tables.nodes.time = stime_l

In [145]:
tables.sort()
ntseq = tables.tree_sequence()